# 05 · 토큰 프루닝 — 문장 안에서 단어를 버리기

앞의 네 노트북은 **덩어리 단위**로 다뤘습니다. 조각을 통째로 남기거나, 버리거나, 치워뒀습니다.
이번에는 **문장 안쪽으로 들어가 단어를 지웁니다.**

```
원문     제7조(청약 철회) 결제 취소 시 결제금액의 10%를 위약 상당액으로 공제합니다.
프루닝   제7조(청약 철회) 결제금액의 10%를 위약 상당액으로 공제합니다.
```

"토큰 압축"이라는 말이 원래 가리키던 기법이고, LLMLingua 계열의 핵심입니다.

### 원리는 한 줄입니다

> **정보량이 적은 단어부터 버립니다.**

문제는 "정보량이 적다"를 어떻게 판단하느냐입니다.
LLMLingua 는 작은 언어모델로 각 토큰의 확률을 계산하지만, 여기서는 **빈도**로 근사합니다.
GPU 도 라이브러리도 필요 없고, 원리는 같습니다.

### 이번에 처음 해보는 것 — 압축률 스윕

지금까지는 점을 하나씩만 찍었습니다. 03번은 `KEEP=3`, 04번은 `keep=5` 한 번씩이었죠.
**"3턴이면 되더라"** 는 알아도 **"2턴이면? 5턴이면?"** 은 몰랐습니다.

이번에는 압축 강도를 100% → 30% 로 훑으면서 매번 측정합니다.
그러면 **어디서 무너지는지** 곡선이 보입니다.

## 0. 이 노트북이 하는 일

**한 문장으로** — 약관 문서에서 단어를 조금씩 더 지워가며,
**어느 지점에서 답이 틀리기 시작하는지** 찾습니다.

### 순서

| # | 하는 일 |
|---|---|
| 1 | 단어마다 **정보량 점수**를 매깁니다 |
| 2 | 점수 낮은 순으로 지워봅니다 |
| 3 | 지우는 양을 **100% → 30% 로 훑으며** 품질을 잽니다 |
| 4 | **보호 규칙**을 껐다 켜며 비교합니다 |
| 5 | 질문 유형마다 **무너지는 지점이 다른지** 봅니다 |
| 6 | 실제 모델 답변으로 검증합니다 |

### 채점은 `survival` 로 합니다

앞의 노트북들이 계속 남긴 숙제였습니다. 여기서 처음 제대로 씁니다.

> **정답에 꼭 필요한 문자열이 압축 후에도 살아 있는가**를 검사합니다.

```
must_include = ["10%", "132,000", "않습니다"]
              ↓
압축 결과에 이 문자열들이 남아 있는지 확인 → 3/3, 2/3 …
```

**LLM 을 부르지 않으므로 비용이 0입니다.** 그래서 수십 번 스윕해도 부담이 없습니다.
실제 모델 검증은 마지막에 몇 지점만 합니다.

## 준비 1 · 설정

01~04번과 같습니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 실험 환경을 갖춥니다.
#   · measure() 텍스트의 input 토큰을 API 로 실측합니다
#   · ask()     압축된 문서로 질문하고 답을 받습니다
# API 호출: 0회 (준비만)
# ──────────────────────────────────────────────────────────────────────────
import json, math, os, re, shutil, subprocess, time
import urllib.request, urllib.error
from collections import Counter
from pathlib import Path

from dotenv import load_dotenv, find_dotenv

from nbtools import Usage, Price, show_table

load_dotenv(find_dotenv(usecwd=True) or str(Path.cwd() / ".env"), override=False)


def require(name):
    v = os.environ.get(name)
    if not v:
        raise RuntimeError(f"{name} 가 없습니다. `cp .env.example .env` 후 값을 채워주세요.")
    return v


ENDPOINT   = require("AZURE_OPENAI_ENDPOINT").rstrip("/")
DEPLOYMENT = require("AZURE_OPENAI_DEPLOYMENT")

AZ_CANDIDATES = [os.environ.get("AZ_CLI"), shutil.which("az"),
                 "/opt/homebrew/bin/az", "/usr/local/bin/az",
                 str(Path.home() / ".local/bin/az")]


def find_az():
    for c in AZ_CANDIDATES:
        if c and Path(c).exists():
            return c
    raise RuntimeError("az CLI 를 찾지 못했습니다. .env 에 AZ_CLI=/전체/경로/az 를 넣어주세요.")


def auth_headers():
    key = os.environ.get("AZURE_OPENAI_API_KEY")
    if key:
        return {"api-key": key}
    r = subprocess.run([find_az(), "account", "get-access-token",
                        "--scope", "https://cognitiveservices.azure.com/.default", "-o", "json"],
                       capture_output=True, text=True, timeout=90)
    if r.returncode != 0:
        raise RuntimeError(f"az 토큰 발급에 실패했습니다.\n{r.stderr.strip()[:300]}")
    return {"Authorization": "Bearer " + json.loads(r.stdout)["accessToken"]}


HEADERS = auth_headers()


def responses(input_text, **params):
    req = urllib.request.Request(
        f"{ENDPOINT}/openai/v1/responses?api-version=preview",
        data=json.dumps({"model": DEPLOYMENT, "input": input_text, **params}).encode(),
        headers={"Content-Type": "application/json", **HEADERS}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=180) as r:
            return json.loads(r.read().decode())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"HTTP {e.code}: {e.read().decode('utf-8','replace')[:400]}")


def rtext(resp):
    out = []
    for item in resp.get("output", []):
        for c in item.get("content", []):
            if c.get("type") in ("output_text", "text"):
                out.append(c.get("text", ""))
    return "".join(out).strip()


def measure(text):
    r = responses(text, max_output_tokens=16)
    return Usage.from_response(r, model=DEPLOYMENT).input_tokens


def ask(context, question):
    r = responses(f"[약관]\n{context}\n\n질문: {question}\n"
                  f"약관에 근거해 한 문장으로 답하세요. 근거가 없으면 '모름'이라고 답하세요.",
                  max_output_tokens=150, temperature=0)
    return rtext(r), Usage.from_response(r, model=DEPLOYMENT)


print("배포:", DEPLOYMENT, "· 준비 완료")

## 준비 2 · 문서와 정답 기준

약관 15개 조항입니다. 실제 약관처럼 **"않습니다 / 없습니다" 가 반복**되게 썼습니다.
이게 뒤에서 중요한 역할을 합니다.

질문마다 **`must_include`** 를 정해 둡니다. 압축 후에도 이 문자열이 살아 있어야 정답을 말할 수 있습니다.

| 유형 | 질문 수 | 살아남아야 할 것 |
|---|---|---|
| **숫자** | 3 | `10%`, `22.53`, `132,000` |
| **부정어** | 3 | `않습니다`, `없습니다` — 이게 사라지면 **뜻이 반대가 됩니다** |
| **주제** | 1 | `고객센터` |

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 약관 문서와 질문별 정답 기준(must_include)을 정의합니다.
#   · 부정어가 반복되는 문서입니다 — 빈도 기반 점수가 낮게 보는 함정
#   · 질문은 숫자·부정어·주제 세 유형으로 나눠 뒤에서 유형별 붕괴 지점을 봅니다
# API 호출: 1회 (원문 토큰 측정)
# ──────────────────────────────────────────────────────────────────────────
DOC = """제1조(목적) 본 약관은 회사가 제공하는 이동통신 서비스의 이용 조건과 절차를 정함을 목적으로 합니다.
제2조(적용 범위) 본 약관에 명시되지 않은 사항은 관계 법령에 따릅니다.
제3조(약관 변경) 회사는 사전 고지 없이 약관을 변경하지 않습니다.
제4조(가입 자격) 만 19세 미만의 미성년자는 법정대리인의 동의 없이 가입할 수 없습니다.
제5조(명의 변경) 요금이 연체된 회선은 명의변경을 신청할 수 없습니다.
제6조(부가서비스) 부가서비스 해지는 당월에 적용되지 않습니다.
제7조(청약 철회) 결제 취소 시 결제금액의 10%를 위약 상당액으로 공제합니다.
제8조(철회 예외) 결제 후 7일 이내 취소는 위약 상당액을 공제하지 않습니다.
제9조(요금 납부) 이용요금은 매월 25일까지 납부하여야 합니다.
제10조(데이터 초과) 기본 제공량 초과 시 1MB당 22.53원이 부과됩니다.
제11조(일시 정지) 일시 정지 기간에는 기본료가 청구되지 않습니다.
제12조(해지 위약금) 약정 내 해지 시 최대 132,000원의 위약금이 발생합니다.
제13조(위약금 면제) 상위 요금제로 변경하면 위약금은 부과되지 않습니다.
제14조(단말 할부) 해지 시 단말기 할부 잔액은 면제되지 않습니다.
제15조(로밍) 로밍은 출국 전 고객센터에서 신청할 수 있습니다."""

QUERIES = [
    ("숫자",   "환불 시 공제되는 수수료율은 얼마인가요?",          ["10%"]),
    ("숫자",   "데이터 초과 시 1MB당 요금은 얼마인가요?",          ["22.53"]),
    ("숫자",   "약정 내 해지 시 최대 위약금은 얼마인가요?",        ["132,000"]),
    ("부정어", "결제 후 7일 이내 취소하면 위약 상당액을 공제하나요?", ["7일", "않습니다"]),
    ("부정어", "상위 요금제로 변경하면 위약금이 부과되나요?",       ["않습니다"]),
    ("부정어", "미성년자가 혼자 가입할 수 있나요?",                ["19세", "없습니다"]),
    ("주제",   "로밍은 어디서 신청하나요?",                       ["고객센터"]),
]

DOC_TOKENS = measure(DOC)

show_table(
    ["항목", "값"],
    [["조항 수", "15개"], ["문자수", f"{len(DOC):,}자"],
     ["어절 수", f"{len(DOC.split()):,}개"], ["input 토큰", f"{DOC_TOKENS:,}"],
     ["질문 수", f"{len(QUERIES)}개 (숫자 3 · 부정어 3 · 주제 1)"]],
    align=["left", "right"],
    title="실험 대상",
    note="'않습니다' 가 5번, '없습니다' 가 2번 나옵니다. 반복되는 단어라는 점이 중요합니다.",
)

## 1. 단어에 점수 매기기

**"정보량이 적은 단어"** 를 어떻게 고를까요. 세 가지를 씁니다.

| 신호 | 규칙 | 이유 |
|---|---|---|
| **빈도** | 자주 나올수록 점수 ↓ | 문서 전체에 흔한 단어는 정보가 적습니다 |
| **길이** | 길수록 점수 ↑ | 짧은 조사·접속어는 대체로 덜 중요합니다 |
| **불용어** | 목록에 있으면 최하점 | `및`, `등`, `그` 같은 것들 |

LLMLingua 는 여기서 **작은 언어모델의 토큰 확률**을 씁니다.
"다음에 올 확률이 높은 토큰 = 예측 가능 = 정보량이 적다" 는 논리입니다.
빈도는 그것의 거친 근사이지만, **버릴 순서를 정한다**는 목적은 같습니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 단어별 정보량 점수를 계산하고 실제로 어떻게 갈리는지 봅니다.
#   · 점수 = log(전체어절수 / 등장횟수) + 길이보너스
#   · 불용어는 최하점으로 밀어 가장 먼저 버려지게 합니다
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
STOPWORDS = {
    "본", "및", "등", "그", "이", "저", "수", "것", "함을", "하는", "있는", "위한",
    "관한", "대한", "따라", "통해", "때", "각", "해당", "관련", "제공하는", "정함을",
    "합니다.", "있습니다.", "됩니다.", "후", "시", "내", "또는", "만",
}


def info_score(words):
    """어절마다 정보량 점수를 매깁니다. 낮을수록 먼저 버려집니다."""
    freq = Counter(words)
    n = len(words)
    out = {}
    for w in set(words):
        if w in STOPWORDS:
            out[w] = -999.0                       # 불용어는 최우선 제거
        else:
            rarity = math.log(n / freq[w])        # 드물수록 큼
            length = min(len(w), 6) / 6           # 길수록 큼 (최대 1.0)
            out[w] = rarity + length
    return out


words = DOC.split()
scores = info_score(words)
ranked = sorted(scores.items(), key=lambda x: x[1])

show_table(
    ["순위", "어절", "점수", "등장", "판정"],
    [[str(i + 1), w, f"{s:.2f}", f"{words.count(w)}회",
      "먼저 버려짐" if s < 1.5 else ""]
     for i, (w, s) in enumerate(ranked[:6])]
    + [["…", "…", "…", "…", ""]]
    + [[str(len(ranked) - 5 + i), w, f"{s:.2f}", f"{words.count(w)}회", "끝까지 남음"]
       for i, (w, s) in enumerate(ranked[-5:])],
    align=["right", "left", "right", "right", "left"],
    title="정보량 점수 — 낮은 쪽이 먼저 버려집니다",
    note="자주 나오는 단어일수록 아래로 갑니다. 여기에 함정이 하나 있는데 4절에서 드러납니다.",
)

## 2. 실제로 지워봅니다

점수가 낮은 순으로 지우고, **남은 어절은 원문 순서대로** 이어 붙입니다.
순서를 바꾸면 모델이 읽기 어려워지기 때문입니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 프루닝 함수를 만들고 한 지점(60%)에서 결과를 눈으로 확인합니다.
#   · rate = 남길 비율. 0.6 이면 어절의 60%만 남깁니다
#   · 지운 뒤에도 원문 순서는 유지합니다
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
PROTECT = re.compile(r"\d|%|[A-Za-z]{2,}|않|없|미만|초과|제외|단,")


def prune(text, rate, protect=True):
    """정보량이 낮은 어절부터 버립니다.

    rate    : 남길 비율 (1.0 = 원문 그대로)
    protect : True 면 숫자·부정어·식별자를 절대 버리지 않습니다
    """
    words = text.split()
    sc = info_score(words)
    if protect:
        sc = {w: (999.0 if PROTECT.search(w) else s) for w, s in sc.items()}

    keep_n = max(1, int(len(words) * rate))
    keep = set(sorted(range(len(words)), key=lambda i: -sc[words[i]])[:keep_n])
    return " ".join(w for i, w in enumerate(words) if i in keep)


def clause(text, no):
    """제N조 부분만 잘라냅니다. 프루닝된 텍스트에서도 동작합니다."""
    i = text.find(f"제{no}조")
    if i < 0:
        return "(조항이 통째로 사라졌습니다)"
    j = text.find(f"제{no+1}조", i)
    return " ".join((text[i:j] if j > 0 else text[i:]).split())


for r in (1.0, 0.6, 0.4):
    t = prune(DOC, r) if r < 1.0 else DOC
    tag = "원문" if r == 1.0 else f"{r:.0%} 프루닝"
    print(f"[{tag}]  전체 {len(t):,}자")
    print(f"  제8조 → {clause(t, 8)}")
    print(f"  제12조 → {clause(t, 12)}\n")

print("남긴 어절은 원문 순서대로 이어 붙입니다. 순서를 섞으면 모델이 읽기 어려워집니다.")

## 3. 압축률 스윕 — 곡선을 그립니다

여기가 이번 노트북의 핵심입니다.

남길 비율을 **100% 에서 30% 까지** 낮춰가며 매번 `survival` 을 잽니다.
LLM 을 부르지 않으므로 **비용 0** 입니다.

```
must_include 가 압축 결과에 남아 있는가?
   "10%"      → 있음
   "않습니다"  → 없음   ← 이 순간 그 질문은 답할 수 없게 됩니다
```

> **범위를 충분히 낮게 잡아야 합니다.**
> 60% 까지만 훑으면 "아무 문제 없다" 는 결론이 나옵니다.
> 벼랑이 그보다 아래에 있으면 못 보고 지나칩니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 압축률을 훑으며 survival 과 토큰 수를 함께 잽니다.
#   · survival: 정답에 필요한 문자열이 살아남은 비율 (LLM 호출 없음)
#   · 토큰은 몇 지점만 실측하고 나머지는 문자수 비율로 근사합니다
# API 호출: 5회 (선택한 지점의 토큰만 실측)
# ──────────────────────────────────────────────────────────────────────────
def survival(text, must):
    """정답에 필요한 문자열이 살아남았는지 검사합니다. 비용 0."""
    flat = text.replace(" ", "").replace(",", "")
    return sum(1 for m in must if m.replace(",", "") in flat) / len(must)


# 붕괴 지점을 찾으려면 충분히 낮은 곳까지 훑어야 합니다.
RATES = [1.0, 0.8, 0.6, 0.5, 0.4, 0.3, 0.25, 0.2, 0.15, 0.1]
MEASURE_AT = {1.0, 0.6, 0.4, 0.3, 0.2, 0.1}     # 이 지점만 실제 토큰을 잽니다

sweep = []
for r in RATES:
    p = prune(DOC, r)
    surv = sum(survival(p, must) for _, _, must in QUERIES) / len(QUERIES)
    full = sum(1 for _, _, must in QUERIES if survival(p, must) == 1.0)
    tok = measure(p) if r in MEASURE_AT else round(DOC_TOKENS * len(p) / len(DOC))
    sweep.append(dict(rate=r, text=p, chars=len(p), tok=tok, surv=surv, full=full))
    if r in MEASURE_AT:
        time.sleep(0.2)

show_table(
    ["남길 비율", "문자수", "토큰", "절감", "survival", "완전히 답 가능"],
    [[f"{s['rate']:.0%}", f"{s['chars']:,}", f"{s['tok']:,}",
      f"{1-s['tok']/DOC_TOKENS:.0%}", f"{s['surv']:.0%}",
      f"{s['full']}/{len(QUERIES)}"] for s in sweep],
    align=["right"] * 6,
    title="압축률 스윕 (보호 규칙 ON)",
    note="'완전히 답 가능' 은 must_include 를 하나도 잃지 않은 질문 수입니다. "
         "이 값이 떨어지기 시작하는 지점이 안전 상한입니다.",
)

## 4. 보호 규칙을 껐다 켜봅니다

앞의 스윕에는 **보호 규칙**이 켜져 있었습니다.

```python
PROTECT = re.compile(r"\d|%|[A-Za-z]{2,}|않|없|미만|초과|제외|단,")
```

숫자·퍼센트·영문·부정어가 들어간 어절은 **점수와 무관하게 절대 버리지 않습니다.**

이게 없으면 어떻게 될까요. 1절에서 말한 **함정**이 여기서 드러납니다.

> `않습니다` 는 이 문서에 **5번** 나옵니다. 자주 나오니 빈도 점수가 낮습니다.
> 즉 **부정어가 먼저 버려집니다.**

그러면 `공제하지 않습니다` 가 `공제하지` 로 잘려 **뜻이 반대로** 읽힙니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 보호 규칙 유무를 나란히 비교합니다.
#   · 같은 압축률에서 survival 이 얼마나 벌어지는지 봅니다
#   · 부정어가 실제로 잘려나가는 장면을 원문으로 확인합니다
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
rows = []
for r in RATES:
    on  = prune(DOC, r, protect=True)
    off = prune(DOC, r, protect=False)
    s_on  = sum(1 for _, _, m in QUERIES if survival(on, m) == 1.0)
    s_off = sum(1 for _, _, m in QUERIES if survival(off, m) == 1.0)
    rows.append([f"{r:.0%}", f"{s_on}/{len(QUERIES)}", f"{s_off}/{len(QUERIES)}",
                 "동일" if s_on == s_off else f"{s_on - s_off:+d}"])

show_table(
    ["남길 비율", "보호 ON", "보호 OFF", "차이"],
    rows,
    align=["right", "right", "right", "right"],
    title="보호 규칙의 효과 (완전히 답 가능한 질문 수)",
    note="보호 규칙 없이는 훨씬 일찍 무너집니다.",
)

# 부정어가 잘리는 장면을 직접 봅니다 (clause() 는 2절에서 정의했습니다)
print("\n제8조 원문의 뜻: '7일 이내 취소는 공제하지 **않는다**'\n")
for label, t in (("원문",            DOC),
                 ("60% · 보호 ON",  prune(DOC, 0.6, protect=True)),
                 ("60% · 보호 OFF", prune(DOC, 0.6, protect=False))):
    print(f"[{label:14s}] {clause(t, 8)}")

print("\n제13조 원문의 뜻: '상위 요금제로 바꾸면 위약금이 부과되지 **않는다**'\n")
for label, t in (("원문",            DOC),
                 ("60% · 보호 ON",  prune(DOC, 0.6, protect=True)),
                 ("60% · 보호 OFF", prune(DOC, 0.6, protect=False))):
    print(f"[{label:14s}] {clause(t, 13)}")

print("\n=> 보호 OFF 에서는 '않습니다' 가 사라집니다.")
print("   '공제하지' / '부과되지' 만 남으면 뜻이 정반대로 읽힐 수 있습니다.")

### 왜 이런 일이 생기나

빈도 기반 점수의 **구조적 약점**입니다.

| 단어 | 등장 | 빈도 점수 | 실제 중요도 |
|---|---|---|---|
| `않습니다` | 5번 | **낮음** | **매우 높음** — 없으면 뜻이 뒤집힙니다 |
| `법정대리인의` | 1번 | 높음 | 중간 |

**"자주 나온다 = 덜 중요하다"** 가 항상 맞지는 않습니다.
법률·약관 문서에서 부정어는 자주 나오면서 동시에 결정적입니다.

LLMLingua 도 같은 문제를 압니다. 그래서 이런 옵션을 제공합니다.

| LLMLingua 옵션 | 이 노트북의 대응 |
|---|---|
| `force_reserve_digit=True` | `PROTECT` 의 `\d` |
| `force_tokens=[...]` | `PROTECT` 의 `않\|없\|제외\|단,` |

**기본값이 `False` 라는 점이 중요합니다.** 켜지 않으면 숫자가 잘려나갑니다.

## 5. 질문 유형마다 무너지는 지점이 다릅니다

전체 평균만 보면 놓치는 게 있습니다. **어떤 질문이 먼저 무너지는지** 나눠서 봅니다.

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 질문 유형별로 survival 곡선을 나눠 봅니다.
#   · 평균은 특정 유형이 0% 인 것을 가립니다
#   · 유형마다 안전 상한이 다르다면 파이프라인도 유형별로 달라야 합니다
# API 호출: 0회
# ──────────────────────────────────────────────────────────────────────────
KINDS = ["숫자", "부정어", "주제"]

rows = []
for r in RATES:
    p = prune(DOC, r)
    row = [f"{r:.0%}"]
    for k in KINDS:
        qs = [(q, m) for kind, q, m in QUERIES if kind == k]
        ok = sum(1 for _, m in qs if survival(p, m) == 1.0)
        row.append(f"{ok}/{len(qs)}")
    rows.append(row)

show_table(
    ["남길 비율"] + KINDS,
    rows,
    align=["right"] + ["right"] * len(KINDS),
    title="유형별 survival (보호 규칙 ON)",
    note="유형마다 버티는 정도가 다릅니다. 가장 약한 유형이 전체 안전 상한을 결정합니다.",
)

# 안전 상한 = 모든 유형이 만점인 가장 낮은 비율
safe = None
for s in sweep:
    if s["full"] == len(QUERIES):
        safe = s["rate"]
print(f"\n모든 질문이 답 가능한 최저 비율: {safe:.0%}")
print(f"그때 토큰: {next(s['tok'] for s in sweep if s['rate']==safe):,} "
      f"(원문 {DOC_TOKENS:,} 대비 {1-next(s['tok'] for s in sweep if s['rate']==safe)/DOC_TOKENS:.0%} 절감)")
print("\n실무에서는 여기서 한 단계 여유를 둡니다. 데이터가 조금만 달라져도 무너질 수 있기 때문입니다.")

## 6. 실제 모델 답변으로 검증

`survival` 은 **문자열이 남아 있는지**만 봅니다. 남아 있어도 모델이 못 읽어낼 수 있고,
반대로 없어도 문맥으로 맞힐 수 있습니다.

그래서 마지막에 **몇 지점만 골라** 실제로 물어봅니다.
이것이 `survival` 을 쓰는 이유이기도 합니다 — **무료로 후보를 좁히고, 유료 검증은 최소로.**

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# 목적: 선택한 압축률에서 실제 모델 답변 정확도를 잽니다.
#   · survival 로 좁힌 지점만 검증합니다 (전 구간을 다 재면 비용이 큽니다)
#   · survival 과 실제 정확도가 얼마나 일치하는지 봅니다
# API 호출: 질문 7개 × 4지점 = 28회 · 약 1분
# ──────────────────────────────────────────────────────────────────────────
CHECK_AT = [1.0, 0.3, 0.2, 0.15]   # 붕괴가 일어나는 구간 위주로

rows = []
for r in CHECK_AT:
    p = prune(DOC, r)
    ok = 0
    for _, q, must in QUERIES:
        ans, _ = ask(p, q)
        flat = ans.replace(" ", "").replace(",", "")
        if all(m.replace(",", "") in flat for m in must):
            ok += 1
        time.sleep(0.15)
    s = next(x for x in sweep if x["rate"] == r)
    rows.append([f"{r:.0%}", f"{s['tok']:,}", f"{1-s['tok']/DOC_TOKENS:.0%}",
                 f"{s['full']}/{len(QUERIES)}", f"{ok}/{len(QUERIES)}",
                 "일치" if ok == s["full"] else f"{ok - s['full']:+d}"])

show_table(
    ["남길 비율", "토큰", "절감", "survival 예측", "실제 정답", "차이"],
    rows,
    align=["right"] * 6,
    title="survival(무료) vs 실제 모델 답변(유료)",
    note="둘이 대체로 일치하면, 앞으로 스윕은 survival 만으로 해도 됩니다. "
         "이것이 골든셋과 survival 지표를 만드는 이유입니다.",
)

## 7. 정리

### 배운 것

- **프루닝은 "정보량이 적은 순서로 버리기"** 입니다. 판단 기준만 다를 뿐 원리는 같습니다
- **빈도 기반 점수에는 구조적 약점이 있습니다** — 자주 나오는 부정어를 먼저 버립니다
- **보호 규칙이 필수입니다** — 숫자·부정어·식별자는 점수와 무관하게 지켜야 합니다
- **압축률과 품질은 비례하지 않습니다** — 어느 지점까지 버티다 갑자기 무너집니다
- **유형마다 무너지는 지점이 다릅니다** — 가장 약한 유형이 전체 상한을 정합니다
- **점 하나만 재면 위험합니다** — 벼랑 끝인지 알 수 없습니다

### 다섯 노트북을 관통하는 것

| 노트북 | 무엇을 | 되돌리기 |
|---|---|---|
| 02 형식 압축 | 형식만 바꿉니다 | 가능 |
| 03 대화 요약 | 오래된 것을 버립니다 | 불가 |
| 04 참조 핸들 | 딴 데 치워둡니다 | 가능 |
| **05 프루닝** | **문장 안 단어를 버립니다** | **불가** |

### LLMLingua 로 넘어갈 때

| 이 노트북 | LLMLingua |
|---|---|
| 빈도 기반 점수 | 작은 LM 의 토큰 확률(self-information) |
| `rate` (남길 비율) | `rate` — **이름도 같습니다** |
| `PROTECT` 의 `\d` | `force_reserve_digit=True` (**기본값 False**) |
| `PROTECT` 의 부정어 | `force_tokens=[...]` |
| — | `question` 을 주면 질문 조건부로 압축 (LongLLMLingua) |

> **주의** — LLMLingua 는 `target_token` 을 주면 `rate` 를 무시합니다. 둘 중 하나만 쓰세요.
> 실제 라이브러리 실험은 `labs/01-llmlingua/` 소관입니다.

### 남은 숙제

- 지금은 `must_include` 를 코드 안에 적었습니다. 케이스가 늘면 **골든셋 파일로 분리**해야 합니다
- 문서가 하나뿐입니다. **여러 도메인**에서 안전 상한이 같은지 확인해야 합니다
- 프루닝은 **캐시를 깹니다**(텍스트가 바뀌므로). 04번처럼 뒤쪽에만 적용하는 편이 안전합니다